<a href="https://colab.research.google.com/github/avs20/infersio/blob/main/20261121_mygpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

here is my plan. I have to create an inference engine and I know nothing about gpt.So I will try to create that.

An inference enginer does fast inference on token preidction /output. So the first step is to load the model. how do we load the model.

let's see hugging face how it mentions to use the model

In [ ]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[{'generated_text': "Hello, I'm a language model, I'm a language model. In my mind, I'm doing the same thing as you. All these different"},
 {'generated_text': 'Hello, I\'m a language model, and I\'m trying to understand what you\'re saying. In English, I have two English words: "l'},
 {'generated_text': "Hello, I'm a language model, so I don't get much of a chance to interact with objects. But I did find that I can do"},
 {'generated_text': 'Hello, I\'m a language model, a language model, a language model."'},
 {'generated_text': "Hello, I'm a language model, not a toolkit.\n\nIn a nutshell, I need to give language models a set of properties that"}]

There is something called as pipeline and generator what are they

https://huggingface.co/docs/transformers/en/main_classes/pipelines

so pipelines hide the internals of how the inference happens. We provide a task. here it's text-generation and model. and then it hides and does the inference.

my questions
1. Is this happening on hf hub or on my instance.
2. need to look into how pipeline is doing. but I cannot look code directly. I ahve to build it step by step.

So the pipine function returns a pipeline object which contains

```
Pipeline

A suitable pipeline for the task.

Utility factory method to build a Pipeline.

A pipeline consists of:

One or more components for pre-processing model inputs, such as a tokenizer, image_processor, feature_extractor, or processor.
A model that generates predictions from the inputs.
Optional post-processing steps to refine the model’s output, which can also be handled by processors.
While there are such optional arguments as `tokenizer`, `feature_extractor`, `image_processor`, and `processor`, they shouldn't be specified all at once. If these components are not provided, `pipeline` will try to load required ones automatically. In case you want to provide these components explicitly, please refer to a specific pipeline in order to get more details regarding what components are required.

```

So basically we have to have one or more components here.

What components we need for gpt2 pipeline.


we need to load a model and then pass the input to the model.
The model takes tokens and we cannot just load a model, we need it's class and how the network is there. It's like seriliazaiton in java or other language.

models is binary file and we need to load the weights to a modal class which can do the multiplication.

from the hugging face we already have a gpt2 model class and a tokenizer.

```
from transformers import GPT2Tokenizer, GPT2Model
```

Let's see the tokenizer first, what it takes as input and what's the output.

https://github.com/huggingface/transformers/blob/v5.23.0/src/transformers/models/gpt2/tokenization_gpt2.py#L31

In [ ]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer

so it's loads the vocab and some merges.txt and it has some tokenizer.json
I think it's the model file . can i see it ?

In [ ]:
tokenizer("Hello world")["input_ids"]

[15496, 995]

In [ ]:
tokenizer("My name is Ashutosh Singh kya haal chal")

{'input_ids': [3666, 1438, 318, 7844, 315, 3768, 14403], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer("My name is Ashutosh Singh kya haal chal")

{'input_ids': [3666, 1438, 318, 7844, 315, 3768, 14403, 479, 3972, 387, 282, 442, 282], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

what is attention mast here and where it is used? I see it's used in GPT forward pass code but will come to it later.

At this point, we give a string to tokenizer and it gets the tokens. ok

now we need to load the model

model can be loaded in 2 ways. one is hugging face and one is loading the model file locally and then loading it from disk. I would start with hugging face first.

In [ ]:
from transformers import GPT2Model

model = GPT2Model.from_pretrained('gpt2')
model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
)

What is see above is models are loaded weights into the class. and we have the model architecture also. we have a forward method which takes the input ids and attention mask to do inference. let's see what it does

In [ ]:
inputs = tokenizer("Hi how are you!")
model.forward(*inputs)

AttributeError: 'str' object has no attribute 'size'

It seems it's wrong forward pass does not generate tokens it needs other parameters

```
inputs = tokenizer("Once upon a time, there was a magical forest", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100)
```

we have to use model directly as a class object and above code for new versions


In [ ]:
model(**inputs)

AttributeError: 'list' object has no attribute 'size'

it does not take direct inputs. it needs pt something tensors
Edit : pt is pytorch so it return pytorch tensors

In [ ]:
input = tokenizer("hello how are you ", return_tensors='pt')
input
model(**input)

BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=tensor([[[-6.1729e-02,  6.7113e-04, -3.0058e-01,  ..., -1.6715e-01,
          -3.8294e-03, -1.4094e-01],
         [ 1.7172e-01, -3.8098e-01, -5.9623e-01,  ..., -2.5943e-01,
           3.8965e-02, -9.5782e-02],
         [-9.5371e-02,  5.5844e-01, -1.8515e-01,  ..., -4.6259e-01,
           2.6502e-01,  8.9716e-02],
         [-3.7940e-02, -2.9920e-01, -8.5477e-01,  ..., -1.0566e-01,
          -7.7610e-02,  2.3951e-02],
         [ 2.6689e-01, -4.8727e-01,  1.4177e+00,  ...,  6.4879e-01,
          -7.0138e-01, -3.4367e-01]]], grad_fn=<ViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

ok this ran but it's of no use to us as of now.
reason : this is core model and gives outputs as logits. We need to post process it and do the work to conver to text.

There are some models but they also do the post processing.
For the sake of the project I will skip this part for now. on how to processing is happening and just use the AutoCasual one to get the output

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

input_ids = tokenizer("Hello, I'm a language model", return_tensors="pt").to(model.device)

output = model.generate(**input_ids)
print(tokenizer.decode(output[0]))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=27) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Hello, I'm a language modeler. I'm a programmer. I'm a writer. I'm a writer. I'm a


In [ ]:
len(output[0])

27

Now it's doing output and a decode. So the first thing is we need to wrap thes 2 steps to a function which does the encoding, generate and decoding

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")


def generate(input_text):

  input_ids = tokenizer(input_text, return_tensors="pt").to(model.device)
  output = model.generate(**input_ids)
  return tokenizer.decode(output[0])

generate("Hello my name is ")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'Hello my name is _____. I am a man of the people. I am a man of the people. I am'

the reason we did this is because we now have a inference method. which can be run when an api is called. It's not streaming but fine. We need to see how much it takes to output.

In [ ]:
generate("Hello my name is")

'Hello my name is John. I am a retired Army Ranger from the United States Army. I am a retired Marine Corps'

what i see is the output is same everytime I think that's because we are not doing sampling. Let's see what does model.generate function does ?


yes the default is greedy approach which is just take the highest probablity token, then we have sampling and beam search. sampling picks from the probabilty distribution of the vocab and the generate method is a for loop.

but in the end it's a for loop

```
    # Simple greedy decoding loop
    while cur_length < max_length:
        logits = model(input_ids).logits
        next_token_logits = logits[:, -1, :]
        next_tokens = torch.argmax(next_token_logits, dim=-1)
        input_ids = torch.cat((input_ids, next_tokens[:, None]), dim=-1)
        cur_length += 1
```

so it get ths logits, get the next token, add the next token to input ids and then generate again. I think we are seeing the auto regressive model here.
1 token at a time.

I would have to explore this.
- why logits [:,-1,:] means
- argmax will chose the location with maxium logit value, but what does dim=-1 does
- what happens when we get a stop token. why do we keep generating. these kind of things.


# 21-11-2026
find the answer to questions myself




In [1]:
from transformers import GPT2LMHeadModel
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

model = GPT2LMHeadModel.from_pretrained('gpt2')



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
input = tokenizer("hello how are you ", return_tensors='pt')
output = model(**input)
output

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[ -37.0707,  -36.4855,  -40.3520,  ...,  -46.5167,  -45.4142,
           -37.9090],
         [ -90.5674,  -90.9249,  -95.4596,  ...,  -98.4348,  -99.4510,
           -92.6363],
         [ -85.1905,  -85.7765,  -88.6023,  ...,  -89.0734,  -92.1447,
           -86.3579],
         [ -99.3057, -101.5496, -106.5623,  ..., -109.5575, -111.0373,
          -104.4708],
         [ -56.2853,  -57.2351,  -58.7746,  ...,  -68.5926,  -65.7597,
           -60.9366]]], grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [3]:
output.logits.shape

torch.Size([1, 5, 50257])

So I see here.
- 1  - input,  we can have multiple
- 5 -  what is 5 ?
- 50257 - this is the vocab size of gpt2. so we have probs of each word in the vocab. We can take the max and get the next token for greedy generation

from the docs

> logits (torch.Tensor of shape (batch_size, sequence_length, config.vocab_size)) — Prediction scores of the language modeling head (scores for each vocabulary token before SoftMax).


so 5 was input size. Let me try different . but I had 4 why 5. oh. Space is treated as a token
let me see both

In [4]:
input = tokenizer("hello how are you", return_tensors='pt')
output = model(**input)
output.logits.shape

torch.Size([1, 4, 50257])

see without space at end. it's 4 tokens

In [5]:
input = tokenizer("Hello my name is Ashutosh Singh ", return_tensors='pt')
output = model(**input)
output.logits.shape

torch.Size([1, 9, 50257])

We cleared it up. input , sequenece and logits

now the question

why logits [:,-1,:] means?

In [6]:
print(output.logits)
print(output.logits[:,-1,:]) # it seems to select the last sequence of the logits, so  there y

tensor([[[-35.2362, -35.3266, -38.9753,  ..., -44.4645, -43.9974, -36.4580],
         [-74.1302, -75.3139, -79.5632,  ..., -81.1064, -82.6454, -78.4388],
         [-54.6548, -56.0921, -62.8933,  ..., -66.3085, -68.8439, -60.8408],
         ...,
         [-57.1195, -58.7315, -65.4999,  ..., -72.6456, -69.5197, -60.3643],
         [-50.5862, -52.0942, -58.4416,  ..., -66.5803, -62.9531, -53.6295],
         [-67.0099, -68.7375, -71.4205,  ..., -81.2542, -77.8336, -73.4131]]],
       grad_fn=<UnsafeViewBackward0>)
tensor([[-67.0099, -68.7375, -71.4205,  ..., -81.2542, -77.8336, -73.4131]],
       grad_fn=<SelectBackward0>)


so it seems to pick the last word of the logits ok. so that is the last layer of the output and we have to get the token from that.

this is the probability before softmax. Which token has the highest value. Got it.

> torch.argmax(next_token_logits, dim=-1)

In [7]:
import torch
next_token_logits = output.logits[:,-1,:]
idx = torch.argmax(next_token_logits, dim= 1)
print(idx)
print(next_token_logits[0][idx])
print(next_token_logits)

tensor([1849])
tensor([-59.7895], grad_fn=<IndexBackward0>)
tensor([[-67.0099, -68.7375, -71.4205,  ..., -81.2542, -77.8336, -73.4131]],
       grad_fn=<SelectBackward0>)


So dim is like row wise. for all the logits .

Let's decode this

In [8]:
tokenizer.decode([1849])

'\xa0'

So this is space. Good point. now we add it to the tokens and run the model again  

In [14]:
cur_length = 0
max_length = 100

input = tokenizer("Hello my name is Ashutosh Singh ", return_tensors='pt')

input_ids = input["input_ids"]
# print(input_ids)
# output.logits.shape
# print(output.logits.shape)

# Simple greedy decoding loop
while cur_length < max_length:
        logits = model(input_ids).logits
        next_token_logits = logits[:, -1, :]
        next_tokens = torch.argmax(next_token_logits, dim=-1)
        input_ids = torch.cat((input_ids, next_tokens[:, None]), dim=-1)
        cur_length += 1

# now I need to find the text decoded
tokenizer.decode(input_ids)

tensor([[15496,   616,  1438,   318,  7844,   315,  3768, 14403,   220]])


['Hello my name is Ashutosh Singh \xa0and I am a student of the Indian Institute of Technology (IIT) in Bangalore. I am a member of the Indian Institute of Technology (IIT) and I am a member of the Indian Institute of Technology (IIT) in Bangalore.\nI am a member of the Indian Institute of Technology (IIT) and I am a member of the Indian Institute of Technology (IIT) in Bangalore.\nI am a member of the Indian Institute of Technology (IIT) and']

ok so it's working now. the issue is it takes almost a minute or more in total. So I need to find how much time each iteration is taking and total time it's taking.

In [15]:
import time

cur_length = 0
max_length = 100

loop_start = time.time()
input = tokenizer("Hello my name is Ashutosh Singh ", return_tensors='pt')

input_ids = input["input_ids"]
# print(input_ids)
# output.logits.shape
# print(output.logits.shape)

# Simple greedy decoding loop
while cur_length < max_length:
        it_start = time.time()
        logits = model(input_ids).logits
        next_token_logits = logits[:, -1, :]
        next_tokens = torch.argmax(next_token_logits, dim=-1)
        input_ids = torch.cat((input_ids, next_tokens[:, None]), dim=-1)
        cur_length += 1
        print(f"it time : {time.time() - it_start}")


print(f"Total time : {time.time() - loop_start}")
# now I need to find the text decoded
tokenizer.decode(input_ids)

it time : 0.16920924186706543
it time : 0.1510934829711914
it time : 0.1575911045074463
it time : 0.17044663429260254
it time : 0.18479609489440918
it time : 0.1731114387512207
it time : 0.17548418045043945
it time : 0.1760265827178955
it time : 0.18627095222473145
it time : 0.19685029983520508
it time : 0.1741657257080078
it time : 0.18174505233764648
it time : 0.1819617748260498
it time : 0.18746447563171387
it time : 0.19042730331420898
it time : 0.20245742797851562
it time : 0.19342303276062012
it time : 0.1968705654144287
it time : 0.19585490226745605
it time : 0.2066035270690918
it time : 0.22128939628601074
it time : 0.2141094207763672
it time : 0.21035313606262207
it time : 0.21394610404968262
it time : 0.22022724151611328
it time : 0.23032140731811523
it time : 0.22672772407531738
it time : 0.2259070873260498
it time : 0.2366023063659668
it time : 0.2500159740447998
it time : 0.2945363521575928
it time : 0.3760335445404053
it time : 0.3928816318511963
it time : 0.3971204757690

['Hello my name is Ashutosh Singh \xa0and I am a student of the Indian Institute of Technology (IIT) in Bangalore. I am a member of the Indian Institute of Technology (IIT) and I am a member of the Indian Institute of Technology (IIT) in Bangalore.\nI am a member of the Indian Institute of Technology (IIT) and I am a member of the Indian Institute of Technology (IIT) in Bangalore.\nI am a member of the Indian Institute of Technology (IIT) and']

ok it took 40 seconds in total and each iteration the time is going up so getting next values is gettnig costlier as the model has to do more calculation. our ttft is 0.15 seconds and then it goes up and up.

I am seeing the whole process of autoregressive ness. here and I think we can wrap this loop / code in a function and that will be our inference function on the api.

This is not streaming I am not sure how that system will look like but after each token we can stream back to the api over sse interface so it would be something like yield in python which does something. I have to go back to that,

What is the difference between generator and yield ?  let's first wrap this code in a function, This is our barebones code

In [17]:
import time



def generate(input_text, max_tokens=20):

        cur_length = 0
        max_length = max_tokens

        loop_start = time.time()
        input = tokenizer(input_text, return_tensors='pt')

        input_ids = input["input_ids"]
        # Simple greedy decoding loop
        times = []
        while cur_length < max_length:
                it_start = time.time()
                logits = model(input_ids).logits #forward pass
                next_token_logits = logits[:, -1, :] # get the last layer logits or logits of next token
                next_tokens = torch.argmax(next_token_logits, dim=-1)   # get the maximum one
                input_ids = torch.cat((input_ids, next_tokens[:, None]), dim=-1) # attach back to input ids
                cur_length += 1
                times.append(time.time()-it_start) # iteration time

        # now I need to find the text decoded
        output = tokenizer.decode(input_ids)
        return {
                "output" : output,
                "ttft" : times[0],
                "total_time" : sum(times)
        }

generate("My name is Anthony gonsalves")

{'output': ['My name is Anthony gonsalves, and I am a member of the National Association of Chiefs of Police. I am a member of'],
 'ttft': 0.16808032989501953,
 'total_time': 3.651702642440796}

well how does it look now?  

In [18]:
generate("My name is Anthony gonsalves. main", max_tokens = 100)

{'output': ['My name is Anthony gonsalves. main character is a young man who is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military. he is a member of the military'],
 'ttft': 0.2378396987915039,
 'total_time': 39.73435401916504}